In [1]:
#!git clone https://github.com/whyhardt/SPICE.git

In [2]:
# !pip install -e SPICE

In [4]:
import sys
import os

from spice import SpiceEstimator, csv_to_dataset, split_data_along_sessiondim
from spice.precoded import workingmemory

sys.path.append("../../..") 
from weinhardt2026.utils.benchmarking_gru import Model as GRU, training as training_gru
from weinhardt2026.studies.castro2025.benchmarking_castro2025 import Castro2025Model, training as training_castro

# For custom RNN
import torch

# print current path
print("Current path:", os.getcwd())

Current path: /home/daniel/repositories/SPICE/weinhardt2026/studies/castro2025


will be pipeline notebook for castro2025 benchmark model

## Load dataset

Let's load the data first with the `csv_to_dataset` method. This method returns a `SpiceDataset` object which we can use right away 

In [2]:
# Load your data, should be in SPICE/weinhardt2025/data/eckstein2024/eckstein2024.csv
path_file = 'data/eckstein2024_processed.csv'
# headers of csv: participant,block,trial_id,choice,reward,rt,payout_1,payout_2,payout_3,payout_4

dataset = csv_to_dataset(
    file = path_file,
    df_participant_id='participant',
    df_choice='choice',
    df_feedback='reward',
    df_block='block',
    )

# Optional reduced dataset for quick sanity checks
use_reduced_data = False
max_sessions = 8
max_trials = 100

if use_reduced_data:
    xs_small = dataset.xs[:max_sessions, :max_trials]
    ys_small = dataset.ys[:max_sessions, :max_trials]
    dataset = dataset.__class__(xs_small, ys_small, device=dataset.device)

# structure of dataset:
# dataset has two main attributes: xs -> inputs; ys -> targets (next action)
# shape: (n_participants*n_blocks*n_experiments, n_timesteps, features)
# features are (n_actions * action, n_actions * reward, n_additional_inputs * additional_input, time_trial, trials, block_number, experiment_id, participant_id)

# in order to set up the participant embedding we have to compute the number of unique participants in our data 
# to get the number of participants n_participants we do:
n_participants = len(dataset.xs[..., -1].unique())
n_actions = dataset.ys.shape[-1]

print(f"Shape of dataset: {dataset.xs.shape}")
print(f"Number of participants: {n_participants}")
print(f"Number of actions in dataset: {n_actions}")
print(f"Number of additional inputs: {dataset.xs.shape[-1]-2*n_actions-5}")
print(f"Number of sessions: {len(dataset.xs[..., -3].unique())}")

Shape of dataset: torch.Size([4158, 150, 1, 13])
Number of participants: 862
Number of actions in dataset: 4
Number of additional inputs: 0
Number of sessions: 5


In [3]:
test_sessions = (2, 3)  # pick sessions that exist for all participants; adjust if needed
dataset_train, dataset_test = split_data_along_sessiondim(dataset, test_sessions)

## SPICE Setup

## SPICE Training

Let's setup now the `SpiceEstimator` object and fit it to the data! 

We are going to do this in two steps:

1. Without fitting the SINDy coefficients to get the pure RNN performance given the selected architecture. 
2. With fitting SINDy coefficients to get the final performance of the interpretable model

That way we can disentangle the gap between GRU and SPICE w.r.t. architecture and SINDy library 

In [15]:
# fitting without SINDy coefficients

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=workingmemory.SpiceModel,
        spice_config=workingmemory.CONFIG,
        n_actions=n_actions,
        n_participants=n_participants,

        epochs=1000,
        
        sindy_weight=0,
        sindy_library_polynomial_degree=2,
        sindy_pruning_frequency=100,
        sindy_ensemble_pruning=0.05,
        sindy_threshold_pruning=0.01,
        sindy_alpha=0.0001,
        
        verbose=True,
    )
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

100%|██████████████████████████████████████████████████████████████| 1000/1000 [45:39<00:00,  2.74s/it, L(Train)=0.4631577, L(Val,RNN)=0.5458835, Conv=5.74e-03]
Maximum number of training epochs reached.
Model did not converge yet.

Stage 2: SINDy refit


100%|██████████| 1000/1000 [10:50<00:00,  1.54it/s, loss=0.0170049, n_params=59.90+/-0.30]



Training results:
	L(Train, RNN): 0.4631577
	L(Val, RNN):   0.5458835

RNN training finished.
Training took 3391.53 seconds.


In [17]:
path_spice = 'params/spice_castro2025.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=workingmemory.SpiceModel,
        spice_config=workingmemory.CONFIG,
        n_actions=n_actions,
        n_participants=n_participants,
        n_experiments=1,
        
        epochs=1000,
        warmup_steps=200,
        learning_rate=0.01,
        batch_size=None,
        ensemble_size=10,
        
        sindy_weight=0.1,
        sindy_alpha=0.0001,
        sindy_pruning_frequency=100,
        sindy_threshold_pruning=0.01,
        sindy_ensemble_pruning=0.05,
        sindy_library_polynomial_degree=2,

        verbose=True,
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        save_path_spice=path_spice,
    )

In [18]:
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

100%|████████████████████████████████████| 1000/1000 [3:09:28<00:00, 11.37s/it, L(Train)=0.5357025, L(Val,RNN)=0.5909653, L(Val,SINDy)=0.7477934, Conv=5.20e-04]
----------------------------------------------------------------------------------------------------------------------------------------------------------------
SPICE Model (Coefficients: 34):
value_reward_chosen[t+1]     = -0.713 1 + 1.0 value_reward_chosen[t] + 0.518 reward[t] + -0.103 reward[t-1] + -0.104 reward[t-2] + -0.119 reward[t-3] + -0.156 value_reward_chosen*reward[t-1] + -0.109 value_reward_chosen*reward[t-2] + 1.064 reward[t]^2 + 0.376 reward[t]*reward[t-1] + 0.314 reward[t]*reward[t-2] + 0.321 reward[t]*reward[t-3] + -0.098 reward[t-1]^2 + -0.141 reward[t-1]*reward[t-2] + -0.128 reward[t-1]*reward[t-3] + -0.152 reward[t-2]^2 + -0.142 reward[t-2]*reward[t-3] + -0.113 reward[t-3]^2 
value_reward_not_chosen[t+1] = 1.0 value_reward_not_chosen[t] + 0.042 reward[t-2] + 0.027 reward[t-3] + -0.076 reward[t-1]^2 
value_cho

 20%|██        | 201/1000 [06:15<24:52,  1.87s/it, loss=0.0242899, n_params=33.58+/-5.53]

Ensemble confidence filtering:
	value_reward_chosen: 13729 -> 12884 / 18102 (participant, experiment, term) slots
	value_reward_not_chosen: 1603 -> 1284 / 12930 (participant, experiment, term) slots
	value_choice_chosen: 6017 -> 5190 / 12930 (participant, experiment, term) slots
	value_choice_not_chosen: 7598 -> 6710 / 12930 (participant, experiment, term) slots


 30%|███       | 301/1000 [09:22<21:49,  1.87s/it, loss=0.0208623, n_params=33.27+/-5.53]

Ensemble confidence filtering:
	value_reward_chosen: 13703 -> 12545 / 18102 (participant, experiment, term) slots
	value_reward_not_chosen: 1540 -> 1234 / 12930 (participant, experiment, term) slots
	value_choice_chosen: 5909 -> 5121 / 12930 (participant, experiment, term) slots
	value_choice_not_chosen: 7526 -> 6610 / 12930 (participant, experiment, term) slots


...

Ensemble confidence filtering:
	value_reward_chosen: 11811 -> 11461 / 18102 (participant, experiment, term) slots
	value_reward_not_chosen: 1090 -> 926 / 12930 (participant, experiment, term) slots
	value_choice_chosen: 4832 -> 4657 / 12930 (participant, experiment, term) slots
	value_choice_not_chosen: 6198 -> 5918 / 12930 (participant, experiment, term) slots


 90%|█████████ | 901/1000 [28:03<03:04,  1.86s/it, loss=0.0208245, n_params=27.29+/-5.80]

Ensemble confidence filtering:
	value_reward_chosen: 11651 -> 11336 / 18102 (participant, experiment, term) slots
	value_reward_not_chosen: 1020 -> 865 / 12930 (participant, experiment, term) slots
	value_choice_chosen: 4771 -> 4594 / 12930 (participant, experiment, term) slots
	value_choice_not_chosen: 6079 -> 5821 / 12930 (participant, experiment, term) slots


100%|██████████| 1000/1000 [31:08<00:00,  1.87s/it, loss=0.0221234, n_params=26.93+/-5.79]



Training results:
	L(Train, RNN): 0.5357025
	L(Val, RNN):   0.5909653
	L(Val, SINDy): 0.7716649

RNN training finished.
Training took 13242.52 seconds.
Saving SPICE model to params/spice_castro2025.pkl...


In [11]:
estimator.load_spice(path_spice)

In [12]:
# Print example SPICE model for first participant
print("\nExample SPICE model (participant 0):")
estimator.print_spice_model(participant_id=0)


Example SPICE model (participant 0):
value_reward_chosen[t+1]     = 0.056 1 + 0.819 value_reward_chosen[t] + 0.304 reward[t] + 0.248 reward[t-1] + 0.112 reward[t-2] + 0.017 reward[t-3] 
value_reward_not_chosen[t+1] = -0.112 1 + 0.853 value_reward_not_chosen[t] + 0.33 reward[t-1] + 0.096 reward[t-2] + -0.04 reward[t-3] 
value_choice[t+1]            = -0.0 1 + 0.707 value_choice[t] + 0.527 choice[t] + 0.077 choice[t-1] + -0.145 choice[t-2] + -0.247 choice[t-3] 


## Benchmarking

### Castro2025 benchmark model

In [ ]:
# Benchmark model: Castro et al. 2025
cfs = Castro2025Model(
    n_participants=n_participants,
    n_actions=n_actions,
    batch_first=True,
    )

path_cfs = path_spice.replace('spice_', 'cfs_')

In [ ]:
optimizer_cfs = torch.optim.Adam(params=cfs.parameters(), lr=0.01)
epochs = 1000

cfs = training_gru(
    model=cfs,
    optimizer=optimizer_cfs,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
)

torch.save(cfs.state_dict(), path_cfs)

Epoch 1/1000: L(Train): 0.6405168175697327; L(Test): 0.6417853832244873
Epoch 2/1000: L(Train): 0.6394693851470947; L(Test): 0.6410409808158875
Epoch 3/1000: L(Train): 0.6384532451629639; L(Test): 0.6403270363807678
Epoch 4/1000: L(Train): 0.6374680399894714; L(Test): 0.6396424174308777
Epoch 5/1000: L(Train): 0.6365119814872742; L(Test): 0.6389834880828857
Epoch 6/1000: L(Train): 0.6355839967727661; L(Test): 0.6383492946624756
Epoch 7/1000: L(Train): 0.6346824169158936; L(Test): 0.6377385258674622
Epoch 8/1000: L(Train): 0.6338068246841431; L(Test): 0.6371498107910156
Epoch 9/1000: L(Train): 0.6329556703567505; L(Test): 0.636581301689148
Epoch 10/1000: L(Train): 0.6321277022361755; L(Test): 0.6360317468643188
Epoch 11/1000: L(Train): 0.6313210129737854; L(Test): 0.6354992985725403
Epoch 12/1000: L(Train): 0.6305359601974487; L(Test): 0.6349830627441406
Epoch 13/1000: L(Train): 0.6297696232795715; L(Test): 0.6344820261001587
Epoch 14/1000: L(Train): 0.6290220618247986; L(Test): 0.63399

In [ ]:
cfs.load_state_dict(torch.load(path_castro, map_location='cpu'))

<All keys matched successfully>

### GRU Model

In [6]:
gru = GRU(n_actions)

path_gru = path_spice.replace('spice_', 'gru_')

In [7]:
epochs = 1000
optimizer_gru = torch.optim.Adam(gru.parameters(), lr=0.01)

gru = training_gru(
    model=gru,
    optimizer=optimizer_gru,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    ).to(torch.device('cpu'))

torch.save(gru.state_dict(), path_gru)

Epoch 1/1000: L(Train): 1.3898439407348633; L(Test): 1.3490817546844482
Epoch 2/1000: L(Train): 1.3504936695098877; L(Test): 1.3134219646453857
Epoch 3/1000: L(Train): 1.3147623538970947; L(Test): 1.2778619527816772
Epoch 4/1000: L(Train): 1.279054880142212; L(Test): 1.2398723363876343
Epoch 5/1000: L(Train): 1.2413148880004883; L(Test): 1.1977266073226929
Epoch 6/1000: L(Train): 1.1992943286895752; L(Test): 1.1505378484725952
Epoch 7/1000: L(Train): 1.152730107307434; L(Test): 1.098620057106018
Epoch 8/1000: L(Train): 1.1018050909042358; L(Test): 1.0435057878494263
Epoch 9/1000: L(Train): 1.0475428104400635; L(Test): 0.987668514251709
Epoch 10/1000: L(Train): 0.9928133487701416; L(Test): 0.933599591255188
Epoch 11/1000: L(Train): 0.9395380020141602; L(Test): 0.8834301233291626
Epoch 12/1000: L(Train): 0.8900359869003296; L(Test): 0.8386691808700562
Epoch 13/1000: L(Train): 0.8458330035209656; L(Test): 0.8003756999969482
Epoch 14/1000: L(Train): 0.8084606528282166; L(Test): 0.768849909

In [19]:
gru.load_state_dict(torch.load(path_gru, map_location='cpu'))

<All keys matched successfully>

# ANALYSIS

In [8]:
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2026.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals

## General Analysis

In [21]:
# Dataset-specific behavioral analysis placeholder.
# Replace with Eckstein2024-specific columns if needed.

In [ ]:
print("Fitted Castro2025 parameters:")
print("\nBeta_r")
print(cfs.beta_r)
print("\nLapse")
print(cfs.lapse)
print("\nPrior")
print(cfs.prior)
print("\nAlpha exploration rate")
print(cfs.alpha_exploration_rate)
print("\nDecay rate")
print(cfs.decay_rate)
print("\nGamma")
print(cfs.gamma)
print("\nTemperature")
print(cfs.temperature)

Fitted Castro2025 parameters:

Beta_r
tensor([0.6951, 0.6931], grad_fn=<ClampBackward1>)

Lapse
tensor([0.4986, 0.5000], grad_fn=<ClampBackward1>)

Prior
tensor([0.6914, 0.6931], grad_fn=<ClampBackward1>)

Alpha exploration rate
tensor([0.5000, 0.5000], grad_fn=<ClampBackward1>)

Decay rate
tensor([0.5008, 0.5000], grad_fn=<ClampBackward1>)

Gamma
tensor([0.6916, 0.6931], grad_fn=<SoftplusBackward0>)

Temperature
tensor([0.6912, 0.6931], grad_fn=<ClampBackward1>)


## Analysis Model Evaluation

In [19]:
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator,
    benchmark_model=cfs.to(torch.device('cpu')),
    gru_model=gru.eval().to(torch.device('cpu')),
    )

W0401 20:23:52.712000 21317 site-packages/torch/_dynamo/convert_frame.py:964] [0/8] torch._dynamo hit config.recompile_limit (8)
W0401 20:23:52.712000 21317 site-packages/torch/_dynamo/convert_frame.py:964] [0/8]    function: '_uncompiled_forward' (/home/daniel/repositories/SPICE/spice/resources/model.py:170)
W0401 20:23:52.712000 21317 site-packages/torch/_dynamo/convert_frame.py:964] [0/8]    last reason: 0/7: state.stride()[1] == inputs.size()[2]*inputs.size()[3]  # (unknown source state.stride()[1], please file a bug)
W0401 20:23:52.712000 21317 site-packages/torch/_dynamo/convert_frame.py:964] [0/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0401 20:23:52.712000 21317 site-packages/torch/_dynamo/convert_frame.py:964] [0/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html.


Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...
Computing choice probabilities with SPICE model...


,Trial Lik.,(std),n_parameters,(std),NLL,(std),AIC,BIC
Benchmark,0.595005,0.160727,13.000000,0.000000,0.519185,0.282692,1.213768,1.476646
GRU,0.604201,0.158885,1844.000000,0.000000,0.503848,0.288445,25.887119,63.175255
SPICE-RNN,0.576104,0.154035,66252.000000,0.000000,0.551466,0.277642,894.981689,2234.683350
SPICE,0.531476,0.159610,26.926912,5.792544,0.632096,0.314636,1.627495,2.171944


## Analysis coefficient distributions

In [ ]:
analysis_coefficients_distributions(
    spice_model=estimator,
    output_dir='results',
)

Extracting coefficient data...
  Ensemble=1, Participants=2, Experiments=1, Modules=3, Total terms=17

ANALYSIS 1: Ensemble Consistency
  Mean presence agreement: 1.000
  Mean presence rate:      1.000
  Ensemble spread plots saved.
  Ensemble CV heatmaps saved.

ANALYSIS 2: Coefficient Distributions
  Terms with >50% presence: 17 / 17
  Terms with 0% presence:   0 / 17
  Violin plots saved.
  Presence rate bar chart saved.
  Experiment comparison skipped (X=1).
  Sparsity heatmaps saved.

All results saved to: ../data/castro2025/


(                     module                     term  term_index  \
 0       value_reward_chosen                        1           0   
 1       value_reward_chosen      value_reward_chosen           1   
 2       value_reward_chosen                reward[t]           2   
 3       value_reward_chosen              reward[t-1]           3   
 4       value_reward_chosen              reward[t-2]           4   
 5       value_reward_chosen              reward[t-3]           5   
 6   value_reward_not_chosen                        1           0   
 7   value_reward_not_chosen  value_reward_not_chosen           1   
 8   value_reward_not_chosen              reward[t-1]           2   
 9   value_reward_not_chosen              reward[t-2]           3   
 10  value_reward_not_chosen              reward[t-3]           4   
 11             value_choice                        1           0   
 12             value_choice             value_choice           1   
 13             value_choice      

## Analysis Individual Differences

In [ ]:
# analysis_coefficients_individuals(
#     criterion="SomeCriterionColumnInYourDataset",
#     analysis="disc",  # also: "cont"
#     reference="ReferenceGroupFromCriterionColumn",  # only necessary if analysis="disc"
    
#     path_data=path_file,
    
#     spice_model=estimator,
    
#     dir_output='results',
# )

'analysis_coefficients_individuals(\n    criterion="SomeCriterionColumnInYourDataset",\n    analysis="disc",  # also: "cont"\n    reference="ReferenceGroupFromCriterionColumn",  # only necessary if analysis="disc"\n\n    path_data=file,\n\n    spice_model=estimator,\n)'